# Colab_From2D — 병변 패치 캐시 (SAM / rembg)

**역할:** `uploads/Tangerine_2D/` 아래에 올린 귤 이미지 → 병변 마스크 기반 RGBA 패치 + `manifest.json` 을 `output/decal_cache/` 에 생성합니다.

**실행 순서:**
1. 런타임 → 런타임 유형 변경 → **GPU** 선택 (SAM 권장)
2. 아래 셀에서 작업 폴더(`Colab_From2D`)로 이동
3. 의존성 설치
4. **이미지 업로드** (왼쪽 폴더 아이콘 또는 Drive) — 상세는 `UPLOAD_GUIDE.txt` 참고
5. (선택) SAM 체크포인트 다운로드
6. `run_decal_cache.py` 실행

---
### 업로드해야 할 것 (요약)

| 무엇 | 어디에 | 필수? |
|------|--------|-------|
| 클래스별 이미지 폴더 | `uploads/Tangerine_2D/<클래스명>/` 안에 `.jpg` 등 | **필수** |
| `sam_vit_b_01ec64.pth` | `checkpoints/` (또는 노트북에서 wget) | 선택 (없으면 휴리스틱만) |

이미지는 **클래스마다 하위 폴더**가 있어야 합니다 (예: `.../Black_spot/a.jpg`). 바로 `Tangerine_2D` 아래에 이미지를 두면 안 됩니다.

## 1) 작업 디렉터리로 이동

**방법 A — 이 폴더만 zip 으로 Drive 에 올린 경우:** zip 해제 경로에 맞게 `BASE` 를 수정하세요.

**방법 B — 전체 Tangerine 레포를 클론한 경우:** `BASE` 를 레포 안의 `Colab_From2D` 절대 경로로 두세요.

In [ ]:
# Colab 예시: Drive 마운트 후 Tangerine/Colab_From2D 로 이동
# from google.colab import drive
# drive.mount("/content/drive")

import os

# ▼▼▼ 여기만 환경에 맞게 수정 ▼▼▼
BASE = "/content/Colab_From2D"  # 예: "/content/drive/MyDrive/Tangerine/Colab_From2D"
# ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲

os.chdir(BASE)
print("cwd:", os.getcwd())

## 2) 의존성 설치

Colab 기본에 `torch` 가 있어도 `rembg`, `segment_anything`, `opencv-headless` 등은 추가 설치가 필요합니다.

In [ ]:
!pip install -q -r requirements-colab.txt

## 3) (선택) SAM ViT-B 체크포인트 받기

`decal_colab.yaml` 기본값은 `checkpoints/sam_vit_b_01ec64.pth` 입니다.

- 이미 Drive 에 올려 두었으면 이 셀은 **실행하지 않아도** 됩니다.
- 없으면 아래 `wget` 으로 받습니다 (약 375MB).

In [ ]:
from pathlib import Path
import urllib.request

ck_dir = Path("checkpoints")
ck_dir.mkdir(parents=True, exist_ok=True)
ck_path = ck_dir / "sam_vit_b_01ec64.pth"
url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

if not ck_path.is_file():
    print("다운로드 중… (약 375MB)")
    urllib.request.urlretrieve(url, ck_path)
    print("저장:", ck_path.resolve())
else:
    print("이미 있음:", ck_path.resolve())

## 4) 이미지 업로드 (직접)

1. 왼쪽 **폴더** 아이콘 → `uploads` → `Tangerine_2D` 로 이동
2. **클래스 이름**으로 새 폴더 생성 (예: `Black_spot`, `Healthy`)
3. 그 안에 `.jpg` / `.png` 이미지를 업로드

또는 로컬에서 맞춰 둔 `Tangerine_2D` 폴더 전체를 zip 으로 올려 `uploads/` 아래에 맞게 풀기.

**상세·주의사항:** 이 폴더의 `UPLOAD_GUIDE.txt` 를 열어 보세요.

## 5) 데칼 캐시 생성 실행

In [ ]:
!python run_decal_cache.py

## 6) 결과 확인

In [ ]:
from pathlib import Path
import json

man = Path("output/decal_cache/manifest.json")
if man.is_file():
    data = json.loads(man.read_text(encoding="utf-8"))
    print("이미지 수:", len(data.get("images", {})))
    for k in list(data.get("images", {}).keys())[:5]:
        print(" ", k, "->", data["images"][k]["patches"])
else:
    print("manifest 없음 — run_decal_cache.py 가 성공했는지 확인")